# vla-hands · Advanced: LoRA, CompositeGraft, Auto Curriculum

**Assumes**: you understand grafts and training from `01_intro` and `02_medium`.

This notebook covers:

| Section | Topic |
|---------|-------|
| 2 | **LoRA** — adapt 0.5 % of VLM params; compare to frozen-backbone baseline |
| 3 | **Freezing curriculum** — `DEFAULT_CURRICULUM` for best final performance |
| 4 | **CompositeGraft** — multiple action heads from one VLM forward pass |
| 5 | **Auto curriculum** — one-call training with automatic environment selection |
| 6 | **Save / reload / Hub** — checkpointing and HuggingFace Hub upload |
| 7 | Summary and next steps |

**Runtime**: ~30–60 min on a T4 GPU; most cells finish quickly.

## 1. Setup

In [ ]:
!pip install -q git+https://github.com/jerod92/project-h.git@claude/vla-robotic-hands-platform-kGiza
# LoRA support requires peft
!pip install -q "peft>=0.10.0"

In [ ]:
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

from transformers import AutoProcessor, AutoModelForVision2Seq

from vla_hands.grafting.core import (
    VLAGraft,
    VisionBridge,
    GraftConfig,
    JoystickAppendage,
    ButtonAppendage,
)
from vla_hands.training.curriculum import (
    TrainingCurriculum,
    CurriculumConfig,
    QUICK_CURRICULUM,
    DEFAULT_CURRICULUM,
)
from vla_hands.envs import TargetNavEnvironment
from vla_hands.utils.viz import plot_training_curves
from vla_hands.utils.benchmark import BenchmarkSuite
from vla_hands.utils.gif import record_expert_gif, save_rollout_gif

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
vlm = AutoModelForVision2Seq.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
vlm = vlm.to(device)

hidden_dim = vlm.config.hidden_size
vision_dim = VLAGraft.detect_vision_dim(vlm)

print(f"Total VLM params : {sum(p.numel() for p in vlm.parameters()) / 1e6:.0f}M")
print(f"hidden_dim       : {hidden_dim}")
print(f"vision_dim       : {vision_dim}")

## 2. LoRA — Parameter-Efficient VLM Adaptation

Standard training in `01_intro` and `02_medium` only updates the appendage MLP (~50 K params).
The frozen VLM backbone is never touched, which is fast but limits what representations are
available to the action head.

**LoRA** (Low-Rank Adaptation) inserts tiny trainable rank-decomposition matrices into the
attention layers of the VLM:

```
W_output = W_frozen  +  B @ A  *  (alpha / r)
              256M params   ~1.2M params
```

- `r=8` adds roughly **0.5 % trainable parameters** — enough for meaningful domain adaptation
- `alpha / r` is an effective learning-rate multiplier (keeping `alpha = 2 * r` is a safe default)
- After training, `merge_lora()` bakes the low-rank updates back into the weight matrices so
  inference speed is unchanged

In [ ]:
from vla_hands.grafting.lora import LoRAConfig, apply_lora, lora_parameter_count

# Baseline: frozen backbone, trainable appendage only
frozen_appendage = VisionBridge(JoystickAppendage(hidden_dim), vision_dim=vision_dim)
frozen_graft = VLAGraft(vlm, frozen_appendage, config=GraftConfig(feature_extraction="last"))

# LoRA: same appendage, but VLM q_proj / v_proj now also train
vlm_lora = apply_lora(
    vlm,
    LoRAConfig(r=8, target_modules=["q_proj", "v_proj"]),
)
lora_appendage = VisionBridge(JoystickAppendage(hidden_dim), vision_dim=vision_dim)
lora_graft = VLAGraft(vlm_lora, lora_appendage, config=GraftConfig(feature_extraction="last"))

counts = lora_parameter_count(vlm_lora)
print("--- LoRA parameter budget ---")
print(f"  VLM total params       : {counts['total']:,}")
print(f"  LoRA trainable params  : {counts['trainable']:,}  ({counts['pct_trainable']:.2f}% of VLM)")

frozen_trainable = sum(p.numel() for p in frozen_graft.parameters() if p.requires_grad)
lora_trainable   = sum(p.numel() for p in lora_graft.parameters()   if p.requires_grad)
print(f"\n  Frozen graft trainable : {frozen_trainable:,}  (appendage only)")
print(f"  LoRA graft trainable   : {lora_trainable:,}  (appendage + LoRA)")

In [ ]:
env_nav = TargetNavEnvironment()

def train_graft(graft, name, bc_steps=300, rl_steps=50):
    config = CurriculumConfig(
        bc_steps=bc_steps,
        rl_steps=rl_steps,
        bc_early_stop_loss=0.03,
        rl_early_stop_success=0.9,
        appendage_lr=3e-4,
        device=device,
        save_dir=f"checkpoints/{name}",
        freezing_stages=QUICK_CURRICULUM,
        eval_every=25,
        log_every=10,
        eval_episodes=5,
    )
    return TrainingCurriculum(graft, processor, env_nav, config).run()

print("Training frozen-backbone baseline...")
metrics_frozen = train_graft(frozen_graft, "frozen_joystick")

print("\nTraining LoRA graft...")
metrics_lora = train_graft(lora_graft, "lora_joystick")

In [ ]:
# Compare training curves side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Frozen backbone vs LoRA — TargetNav", fontsize=13)

for col, (label, metrics) in enumerate(
    [("Frozen backbone", metrics_frozen), ("LoRA (r=8)", metrics_lora)]
):
    src_fig = plot_training_curves(metrics, title=label)
    for line in src_fig.axes[0].lines:
        axes[col].plot(line.get_xdata(), line.get_ydata())
    axes[col].set_title(label)
    axes[col].set_xlabel("BC step")
    axes[col].set_ylabel("Loss")
    axes[col].grid(alpha=0.3)
    plt.close(src_fig)

plt.tight_layout()
plt.show()

# Benchmark comparison
for label, graft in [("Frozen backbone", frozen_graft), ("LoRA", lora_graft)]:
    r = BenchmarkSuite(graft, processor, device=device).run_benchmark(env_nav, n_episodes=20)
    print(f"{label:20s}  success={r.success_rate:.2f}  mean_reward={r.mean_reward:.2f}")

## 3. Freezing Curriculum — Trading Speed for Quality

`QUICK_CURRICULUM` keeps the VLM fully frozen — fastest wall-clock time, but the backbone
representations can never specialise.

`DEFAULT_CURRICULUM` gradually unfreezes VLM layers as training progresses. The idea: give
the appendage time to stabilise on the frozen representation, then allow the VLM to drift
toward more action-predictive features.

| Stage | Layers unfrozen | Starts at step |
|---|---|---|
| `appendage_only` | 0 | 0 |
| `last_2` | 2 | 500 |
| `last_6` | 6 | 1 000 |
| `full` | all | 2 000 |

In [ ]:
# Inspect the DEFAULT_CURRICULUM stage list
print("DEFAULT_CURRICULUM stages:")
print(f"  {'stage name':22s}  {'layers unfrozen':>15s}  {'start step':>10s}")
print("  " + "-" * 52)
for stage in DEFAULT_CURRICULUM:
    print(
        f"  {stage.name:22s}  {stage.n_layers_unfrozen:>15d}  {stage.start_step:>10d}"
    )

In [ ]:
# Full curriculum training on a fresh graft — trades speed for better final performance.
# Increase bc_steps to 2500 and rl_steps to 500 for a production-quality run.
full_appendage = VisionBridge(JoystickAppendage(hidden_dim), vision_dim=vision_dim)
full_graft = VLAGraft(vlm, full_appendage, config=GraftConfig(feature_extraction="last"))

full_config = CurriculumConfig(
    bc_steps=600,          # increase to 2500 for best results
    rl_steps=100,          # increase to 500 for best results
    bc_early_stop_loss=0.02,
    rl_early_stop_success=0.95,
    appendage_lr=3e-4,
    device=device,
    save_dir="checkpoints/full_curriculum_joystick",
    freezing_stages=DEFAULT_CURRICULUM,
    eval_every=50,
    log_every=20,
    eval_episodes=5,
)

print("Training with DEFAULT_CURRICULUM...")
metrics_full = TrainingCurriculum(full_graft, processor, env_nav, full_config).run()

r_full = BenchmarkSuite(full_graft, processor, device=device).run_benchmark(env_nav, n_episodes=20)
print(f"\nDEFAULT_CURRICULUM  success={r_full.success_rate:.2f}  mean_reward={r_full.mean_reward:.2f}")

## 4. CompositeGraft — Multiple Action Heads, One Forward Pass

Some tasks require simultaneous control — steer *and* press a button. Running the VLM once
per action head would double inference time. `CompositeGraft` solves this:

```
image + prompt
      |
 [VLM backbone]   <- one forward pass
      |
 last-token hidden state
      |----> [JoystickAppendage]  -> (dx, dy)
      |----> [ButtonAppendage]    -> press confidence
```

`.forward()` returns a `dict` keyed by the names you provided at construction time.

In [ ]:
from vla_hands.grafting.composite import CompositeGraft

composite = CompositeGraft(
    vlm,
    {
        "joystick": JoystickAppendage(hidden_dim),
        "button":   ButtonAppendage(hidden_dim),
    },
    config=GraftConfig(feature_extraction="last"),
)
composite = composite.to(device)

print(f"Appendage heads : {list(composite.appendages.keys())}")
total_head_params = sum(
    p.numel() for a in composite.appendages.values() for p in a.parameters()
)
print(f"Total head params : {total_head_params:,}")

# Dummy forward pass — confirm output dict structure
dummy_input = {"input_ids": torch.zeros(1, 5, dtype=torch.long, device=device)}
with torch.no_grad():
    out = composite(**dummy_input)

print(f"\nForward pass output keys : {list(out.keys())}")
print(f"  joystick shape : {out['joystick'].shape}   # (batch, 2) — dx, dy")
print(f"  button   shape : {out['button'].shape}     # (batch, 1) — press confidence")

In [ ]:
# Train CompositeGraft on TargetNav.
# The joystick head drives movement; the button head acts as an auxiliary signal
# (e.g. confirming target reached). Both heads train jointly from one BC loss sum.

composite.train()
optimizer = torch.optim.AdamW(
    [p for p in composite.parameters() if p.requires_grad], lr=3e-4
)

BC_STEPS = 300
losses = []

print(f"Training CompositeGraft ({BC_STEPS} BC steps)...")
for step in range(BC_STEPS):
    # For TargetNav the expert provides a joystick action; button target is 0.0 by default
    env_nav.reset(seed=step)
    obs = env_nav.reset(seed=step)
    expert_action = env_nav.expert_action()  # returns (dx, dy) array

    from vla_hands.training.trainer import _preprocess
    inputs = _preprocess(processor, obs, env_nav.prompt, device)
    out = composite(**inputs)

    tgt_joy = torch.tensor([expert_action], dtype=torch.float32, device=device)
    tgt_btn = torch.zeros(1, 1, dtype=torch.float32, device=device)

    loss_joy = composite.appendages["joystick"].action_loss(out["joystick"], tgt_joy)
    loss_btn = composite.appendages["button"].action_loss(out["button"], tgt_btn)
    loss = loss_joy + 0.1 * loss_btn   # down-weight auxiliary head

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(composite.parameters(), 1.0)
    optimizer.step()
    losses.append(loss.item())

    if (step + 1) % (BC_STEPS // 6) == 0:
        print(f"  step {step + 1:4d}/{BC_STEPS}  loss={loss.item():.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.title("CompositeGraft BC Loss (joystick + button)")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Accessing both action outputs at inference time
composite.eval()
obs = env_nav.reset(seed=99)

from vla_hands.training.trainer import _preprocess
inputs = _preprocess(processor, obs, env_nav.prompt, device)

with torch.no_grad():
    actions = composite(**inputs)

dx, dy  = actions["joystick"][0].cpu().tolist()
press   = actions["button"][0].item()

print(f"Joystick action : dx={dx:+.3f}  dy={dy:+.3f}")
print(f"Button action   : press={press:.3f}  ({'pressed' if press > 0.5 else 'not pressed'})")

## 5. Auto Curriculum

`auto_curriculum` is the shortest path from a VLM to a trained graft. It:

1. Infers the best environment(s) for the requested appendage type
2. Builds the graft (single `VLAGraft` or `CompositeGraft` depending on the count)
3. Runs BC + RL with a sensible budget split
4. Returns `(graft, metrics)` ready to evaluate

In [ ]:
from vla_hands.training.auto import auto_curriculum

# Single appendage — auto selects environment and trains
print("Running auto_curriculum for joystick...")
auto_graft, auto_metrics = auto_curriculum(
    vlm=vlm,
    processor=processor,
    appendage="joystick",
    device=device,
    bc_steps=300,
    rl_steps=50,
)

print(f"\nReturned type : {type(auto_graft).__name__}")
print(f"BC final loss : {auto_metrics['bc']['loss'][-1]:.4f}")
print(f"RL final SR   : {auto_metrics['rl']['success_rate'][-1]:.2f}")

## 6. Save / Reload / Hub

Checkpoints contain only the **appendage weights** (100 KB – 2 MB). The VLM backbone is
always reloaded from the HuggingFace Hub at `from_pretrained` time, keeping checkpoints tiny.

In [ ]:
import os

# Save
auto_graft.save("checkpoints/auto_joystick_final")

files = os.listdir("checkpoints/auto_joystick_final")
print("Checkpoint files:")
for f in sorted(files):
    size_kb = os.path.getsize(f"checkpoints/auto_joystick_final/{f}") / 1024
    print(f"  {f}  ({size_kb:.1f} KB)")

In [ ]:
# Reload — VLM is pulled from the Hub, appendage weights restored from checkpoint
reloaded = VLAGraft.from_pretrained(
    vlm_id=MODEL_ID,
    appendage=VisionBridge(
        JoystickAppendage(hidden_dim),
        vision_dim=vision_dim,
    ),
    checkpoint_path="checkpoints/auto_joystick_final",
    device=device,
)

print(f"Reloaded: {type(reloaded).__name__}")

r = BenchmarkSuite(reloaded, processor, device=device).run_benchmark(env_nav, n_episodes=10)
print(f"Reloaded graft: success={r.success_rate:.2f}  mean_reward={r.mean_reward:.2f}")

In [ ]:
# Upload to HuggingFace Hub (optional — requires huggingface_hub and a write token)
#
# !pip install -q huggingface_hub
# from huggingface_hub import HfApi
#
# api = HfApi()
# api.upload_folder(
#     folder_path="checkpoints/auto_joystick_final",
#     repo_id="your-username/vla-hands-joystick",
#     repo_type="model",
# )
#
# Then anyone can reload it with:
# VLAGraft.from_pretrained(
#     vlm_id="HuggingFaceTB/SmolVLM-256M-Instruct",
#     appendage=VisionBridge(JoystickAppendage(hidden_dim), vision_dim=vision_dim),
#     checkpoint_path="your-username/vla-hands-joystick",
#     device=device,
# )

print("Uncomment the block above to push your checkpoint to HuggingFace Hub.")

## 7. Summary

You have now covered the full vla-hands stack:

| Topic | Key API |
|---|---|
| LoRA fine-tuning | `apply_lora(vlm, LoRAConfig(r=8, target_modules=[...]))` |
| Param count | `lora_parameter_count(vlm_lora)` |
| Freezing schedule | `CurriculumConfig(freezing_stages=DEFAULT_CURRICULUM, ...)` |
| Multi-head control | `CompositeGraft(vlm, {'joy': ..., 'btn': ...}, config)` |
| One-call training | `auto_curriculum(vlm, processor, 'joystick', ...)` |
| Save/load | `graft.save(path)` / `VLAGraft.from_pretrained(vlm_id, appendage, path)` |

### Ideas for further exploration

- **New appendages**: `SliderAppendage` (1D continuous), chord-key `MultiButtonAppendage`
- **New environments**: Snake (joystick), Simon Says (button sequence), camera pan/tilt
- **Training**: DAgger-style interactive collection, PPO instead of REINFORCE
- **Architecture**: cross-attention over patch embeddings instead of mean-pool skip connection
- **Multi-task BC**: train one graft on three environments simultaneously

Contributions welcome at the GitHub repository.